# 02. Data Cleaning and Processed Dataset Creation

**Project:** Retail Sales Forecasting & Analytics  
**Phase:** Phase 5 – Data Cleaning and Verification  
**Input Data:** `data/raw/Sample - Superstore.csv` (Immutable)  
**Output Data:** `data/processed/superstore_cleaned.csv`  

### Core Principles
1. **Immutability:** The raw CSV is never modified, renamed, or overwritten.
2. **Transparency:** Every transformation must have an empirical business justification.
3. **No Arbitrary Loss:** Negative profit and extreme values are preserved if they represent genuine business reality.
4. **Reproducibility:** All transformations are implemented in reusable modular functions in `src.data_cleaning`.

## 1. Environment and Data Ingestion
We load the raw dataset using `src.data_loader.load_raw_data()` to ensure exact reproducibility.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from src.data_loader import load_raw_data

df_raw = load_raw_data(parse_dates=True)
print(f"Raw dataset shape: {df_raw.shape} (Rows: {len(df_raw):,}, Columns: {df_raw.shape[1]})")
df_raw.head(3)

## 2. Investigation of Duplicate Records
In Phase 4 data validation, 8 `(Order ID, Product ID)` combinations representing 16 rows were identified.
We evaluate whether these are genuine multi-line purchases or true accidental duplicates.

In [ ]:
dup_pairs = df_raw[df_raw.duplicated(subset=['Order ID', 'Product ID'], keep=False)].sort_values(by=['Order ID', 'Product ID'])
print(f"Total rows sharing (Order ID, Product ID): {len(dup_pairs)}")

cols_inspect = ['Row ID', 'Order ID', 'Customer ID', 'Product ID', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
dup_pairs[cols_inspect]

### Duplicate Analysis Findings:
- **7 Pairs (14 rows):** Feature differing quantities, sales, and profits (e.g., Row 6499 has Qty=9, Sales=$135.09 while Row 6501 has Qty=6, Sales=$90.06). These represent legitimate separate line items on the same order (e.g. split orders, separate packages, or staggered shipments). They are **retained**.
- **1 Pair (2 rows: Row ID 3406 & Row ID 3407):** Both records possess identical Order ID (`US-2014-150119`), Customer ID (`LB-16795`), Product ID (`FUR-CH-10002965`), Quantity (`2`), Sales (`$281.372`), Discount (`0.3`), and Profit (`-$12.0588`). Across all 20 business columns, they are exact clones. This is a confirmed accidental double-entry logging duplicate.
- **Decision:** Drop `Row ID 3407` (second occurrence) and retain `Row ID 3406`. Exactly 1 record is removed.

## 3. Investigation of Whitespace in Text Columns
Check for typographical whitespace anomalies across text fields.

In [ ]:
text_cols = df_raw.select_dtypes(include=['object', 'string']).columns
ws_report = {}
for col in text_cols:
    s = df_raw[col].astype(str)
    ws_count = int((s != s.str.strip()).sum())
    if ws_count > 0:
        ws_report[col] = ws_count
        sample_vals = df_raw[s != s.str.strip()][col].head(3).tolist()
        print(f"{col}: {ws_count} rows with whitespace issues. Samples: {sample_vals}")

print("Whitespace summary:", ws_report if ws_report else "All text columns clean")

### Whitespace Decision:
`Product Name` contains 16 rows with trailing whitespace.  
**Decision:** Apply `.str.strip()` to all text fields during cleaning to ensure string standardization and prevent join/filter mismatches.

## 4. Categorical Consistency Verification
Inspect unique domain values and check for casing variations across geographic and operational categories.

In [ ]:
for c in ['Category', 'Segment', 'Region', 'Ship Mode']:
    print(f"{c}: {sorted(df_raw[c].unique().tolist())}")

print(f"Unique States: {df_raw['State'].nunique()}")
print(f"Unique Cities: {df_raw['City'].nunique()}")
print(f"Lowercase City Unique Count: {df_raw['City'].str.lower().nunique()} (Matches original count, indicating 0 casing mismatches)")

## 5. Numerical Range Validation & Profit Distribution
Validate bounds on quantitative variables and examine the negative profit distribution.

In [ ]:
print(df_raw[['Sales', 'Quantity', 'Discount', 'Profit']].describe())

neg_profits = df_raw[df_raw['Profit'] < 0]
print(f"\nNegative profit count: {len(neg_profits)} ({len(neg_profits)/len(df_raw)*100:.2f}%)")
print(f"Mean discount on unprofitable orders: {neg_profits['Discount'].mean():.2%}")
print(f"Mean discount on profitable orders: {df_raw[df_raw['Profit'] >= 0]['Discount'].mean():.2%}")

### Numerical Decision:
- `Sales > 0`: Confirmed (min $0.44).
- `Quantity >= 1`: Confirmed (min 1 unit).
- `0.0 <= Discount <= 1.0`: Confirmed (range 0.0 to 0.8).
- **Negative Profit:** Strongly correlated with steep discounts (average 37.8% vs 10.5% for profitable orders). This reflects genuine commercial loss-leader activity.
- **Decision:** Retain all negative profit transactions and extreme sales values without truncation or removal.

## 6. Date Chronology and Shipping Duration
Verify temporal sequencing across all records.

In [ ]:
shipping_days = (df_raw['Ship Date'] - df_raw['Order Date']).dt.days
print(f"Shipping duration range: {shipping_days.min()} to {shipping_days.max()} days (Mean: {shipping_days.mean():.2f})")
print(f"Chronology violations (Order Date > Ship Date): {(df_raw['Order Date'] > df_raw['Ship Date']).sum()}")

## 7. Run Pipeline and Generate Processed Dataset
Execute the clean_data pipeline imported from `src.data_cleaning`.

In [ ]:
from src.data_cleaning import clean_data, save_processed_data

df_cleaned, summary = clean_data(df_raw)
output_file = save_processed_data(df_cleaned)

print("Cleaning Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

print(f"\nProcessed dataset successfully written to: {output_file}")
df_cleaned.head(3)